# Notebook 2: Link Clinical Notes to Labeled Admissions

## Objective
Pair discharge notes from MIMIC-IV-Note with labeled admissions to create a multimodal dataset combining structured data and clinical text.

## Input Data
- `labeled_admissions_30day_readmission.csv` (518,349 admissions from Notebook 1)
- `discharge.csv` - Discharge summaries from MIMIC-IV-Note dataset

## Process
1. **Load labeled admissions** - Import the 518K admissions with readmission labels from Notebook 1
2. **Load discharge notes** - Import clinical discharge summaries from MIMIC-IV-Note
3. **Linking strategy**:
   - Merge on `hadm_id` (hospital admission ID)
   - Use inner join to keep only admissions that have both labels AND discharge notes
4. **Handle duplicates**:
   - Check for multiple notes per admission
   - If duplicates exist, keep the most recent note (closest to discharge)
5. **Verify data quality**:
   - Check note lengths and coverage
   - Ensure readmission label distribution remains balanced

## Output
- `labeled_admissions_with_notes.csv` (311,459 admissions)
  - Contains: All columns from Notebook 1 PLUS discharge note text
  - Coverage: 60.09% of labeled admissions have discharge notes
  - Readmission rate: 20.65% (slightly higher than original 19.58%)

## Key Findings
- 331,793 unique admissions have discharge notes in MIMIC-IV-Note
- 311,459 admissions have both labels AND notes (60% coverage)
- Lost 206,890 admissions (40%) that don't have discharge notes
- Average note length: ~10,000 characters
- No duplicate notes per admission (each admission has exactly one discharge note)

## Why 40% Loss?
Not all hospital admissions result in discharge summaries being recorded in MIMIC-IV-Note. This is expected and reflects real clinical documentation practices.

## Next Step
Notebook 3 will link timestamped clinical events to these admissions.

In [ ]:
# Mount Google Drive
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [ ]:
import pandas as pd
import numpy as np
from pathlib import Path

pd.set_option('display.max_columns', 50)
print("Libraries imported")

Libraries imported


In [ ]:
DATA_DIR = Path('/content/drive/MyDrive/MIDS/w266/Final Project/data')
OUTPUT_DIR = Path('/content/drive/MyDrive/MIDS/w266/Final Project/output')

In [ ]:
# Load labeled dataset from Step 1
labeled_admissions = pd.read_csv(OUTPUT_DIR / 'labeled_admissions_30day_readmission.csv')

print(f"Loaded {len(labeled_admissions):,} labeled admissions")
print(f"Shape: {labeled_admissions.shape}")
labeled_admissions.head()

Loaded 518,349 labeled admissions
Shape: (518349, 15)


,subject_id,hadm_id,admittime,dischtime,admission_type,admission_location,discharge_location,insurance,language,marital_status,race,los_days,readmitted_30day,days_to_readmission,readmission_hadm_id
0,10000032,22595853,2180-05-06 22:23:00,2180-05-07 17:15:00,URGENT,TRANSFER FROM HOSPITAL,HOME,Medicaid,English,WIDOWED,WHITE,0.786111,0,NaN,NaN
1,10000032,22841357,2180-06-26 18:27:00,2180-06-27 18:49:00,EW EMER.,EMERGENCY ROOM,HOME,Medicaid,English,WIDOWED,WHITE,1.015278,1,25.740278,29079034.0
2,10000032,29079034,2180-07-23 12:35:00,2180-07-25 17:55:00,EW EMER.,EMERGENCY ROOM,HOME,Medicaid,English,WIDOWED,WHITE,2.222222,1,11.242361,25742920.0
3,10000068,25022803,2160-03-03 23:16:00,2160-03-04 06:26:00,EU OBSERVATION,EMERGENCY ROOM,NaN,NaN,English,SINGLE,WHITE,0.298611,0,NaN,NaN
4,10000084,23052089,2160-11-21 01:56:00,2160-11-25 14:52:00,EW EMER.,WALK-IN/SELF REFERRAL,HOME HEALTH CARE,Medicare,English,MARRIED,WHITE,4.538889,0,NaN,NaN


In [ ]:
# Show all columns
print("Columns in labeled dataset:")
print(labeled_admissions.columns.tolist())

Columns in labeled dataset:
['subject_id', 'hadm_id', 'admittime', 'dischtime', 'admission_type', 'admission_location', 'discharge_location', 'insurance', 'language', 'marital_status', 'race', 'los_days', 'readmitted_30day', 'days_to_readmission', 'readmission_hadm_id']


In [ ]:
# Load discharge notes
print("Loading discharge.csv...")
print("This may take a minute - it's a large file...\n")

discharge_notes = pd.read_csv(DATA_DIR / 'discharge.csv')

print(f"Loaded {len(discharge_notes):,} discharge notes")
print(f"Shape: {discharge_notes.shape}")
print(f"\nColumns: {discharge_notes.columns.tolist()}")

Loading discharge.csv...
This may take a minute - it's a large file...

Loaded 331,793 discharge notes
Shape: (331793, 8)

Columns: ['note_id', 'subject_id', 'hadm_id', 'note_type', 'note_seq', 'charttime', 'storetime', 'text']


In [ ]:
# Look at the structure
print("\nFirst discharge note:")
discharge_notes.head(3)


First discharge note:


,note_id,subject_id,hadm_id,note_type,note_seq,charttime,storetime,text
0,10000032-DS-21,10000032,22595853,DS,21,2180-05-07 00:00:00,2180-05-09 15:26:00,\nName: ___ Unit No: _...
1,10000032-DS-22,10000032,22841357,DS,22,2180-06-27 00:00:00,2180-07-01 10:15:00,\nName: ___ Unit No: _...
2,10000032-DS-23,10000032,29079034,DS,23,2180-07-25 00:00:00,2180-07-25 21:42:00,\nName: ___ Unit No: _...


In [ ]:
# Check the text column
print("\nNote text column info:")
print(f"Column name with text: 'text'")
print(f"Average note length: {discharge_notes['text'].str.len().mean():.0f} characters")
print(f"Median note length: {discharge_notes['text'].str.len().median():.0f} characters")
print(f"Max note length: {discharge_notes['text'].str.len().max():.0f} characters")

# Show a sample note (first 500 characters)
print("\nSample note (first 500 chars):")
print("-"*60)
print(discharge_notes['text'].iloc[0][:500])
print("...")


Note text column info:
Column name with text: 'text'
Average note length: 10551 characters
Median note length: 9847 characters
Max note length: 60381 characters

Sample note (first 500 chars):
------------------------------------------------------------
 
Name:  ___                     Unit No:   ___
 
Admission Date:  ___              Discharge Date:   ___
 
Date of Birth:  ___             Sex:   F
 
Service: MEDICINE
 
Allergies: 
No Known Allergies / Adverse Drug Reactions
 
Attending: ___
 
Chief Complaint:
Worsening ABD distension and pain 
 
Major Surgical or Invasive Procedure:
Paracentesis

 
History of Present Illness:
___ HCV cirrhosis c/b ascites, hiv on ART, h/o IVDU, COPD, 
bioplar, PTSD, presented from OSH ED with worsening abd 
d
...


In [ ]:
# Check how many unique admissions have notes
print("\nChecking hadm_id coverage:")
print(f"Unique hadm_ids in labeled admissions: {labeled_admissions['hadm_id'].nunique():,}")
print(f"Unique hadm_ids in discharge notes: {discharge_notes['hadm_id'].nunique():,}")

# Check overlap
labeled_hadm_ids = set(labeled_admissions['hadm_id'])
notes_hadm_ids = set(discharge_notes['hadm_id'])

overlap = labeled_hadm_ids.intersection(notes_hadm_ids)
print(f"\nOverlap: {len(overlap):,} admissions have both labels and notes")
print(f"Coverage: {len(overlap)/len(labeled_hadm_ids)*100:.2f}% of labeled admissions have notes")


Checking hadm_id coverage:
Unique hadm_ids in labeled admissions: 518,349
Unique hadm_ids in discharge notes: 331,793

Overlap: 311,459 admissions have both labels and notes
Coverage: 60.09% of labeled admissions have notes


In [ ]:
# Merge on hadm_id
print("\nMerging labeled admissions with discharge notes...")

# Keep only relevant columns from discharge notes
notes_subset = discharge_notes[['hadm_id', 'note_id', 'charttime', 'storetime', 'text']].copy()

# Merge
labeled_with_notes = labeled_admissions.merge(
    notes_subset,
    on='hadm_id',
    how='inner'  # Only keep admissions that have notes
)

print(f"\nMerged dataset:")
print(f"  Before merge: {len(labeled_admissions):,} labeled admissions")
print(f"  After merge: {len(labeled_with_notes):,} admissions with notes")
print(f"  Lost: {len(labeled_admissions) - len(labeled_with_notes):,} admissions ({(len(labeled_admissions) - len(labeled_with_notes))/len(labeled_admissions)*100:.2f}%)")


Merging labeled admissions with discharge notes...

Merged dataset:
  Before merge: 518,349 labeled admissions
  After merge: 311,459 admissions with notes
  Lost: 206,890 admissions (39.91%)


In [ ]:
# Check if label distribution changed
print("\nLabel distribution after merging with notes:")
print(labeled_with_notes['readmitted_30day'].value_counts())
print("\nProportions:")
print(labeled_with_notes['readmitted_30day'].value_counts(normalize=True))

print(f"\nReadmission rate:")
print(f"  Before merge: {labeled_admissions['readmitted_30day'].mean()*100:.2f}%")
print(f"  After merge: {labeled_with_notes['readmitted_30day'].mean()*100:.2f}%")


Label distribution after merging with notes:
readmitted_30day
0    247150
1     64309
Name: count, dtype: int64

Proportions:
readmitted_30day
0    0.793523
1    0.206477
Name: proportion, dtype: float64

Readmission rate:
  Before merge: 19.58%
  After merge: 20.65%


In [ ]:
# Show the merged dataset structure
print("\nMerged dataset columns:")
print(labeled_with_notes.columns.tolist())

print("\nFirst 3 rows:")
labeled_with_notes.head(3)


Merged dataset columns:
['subject_id', 'hadm_id', 'admittime', 'dischtime', 'admission_type', 'admission_location', 'discharge_location', 'insurance', 'language', 'marital_status', 'race', 'los_days', 'readmitted_30day', 'days_to_readmission', 'readmission_hadm_id', 'note_id', 'charttime', 'storetime', 'text']

First 3 rows:


,subject_id,hadm_id,admittime,dischtime,admission_type,admission_location,discharge_location,insurance,language,marital_status,race,los_days,readmitted_30day,days_to_readmission,readmission_hadm_id,note_id,charttime,storetime,text
0,10000032,22595853,2180-05-06 22:23:00,2180-05-07 17:15:00,URGENT,TRANSFER FROM HOSPITAL,HOME,Medicaid,English,WIDOWED,WHITE,0.786111,0,NaN,NaN,10000032-DS-21,2180-05-07 00:00:00,2180-05-09 15:26:00,\nName: ___ Unit No: _...
1,10000032,22841357,2180-06-26 18:27:00,2180-06-27 18:49:00,EW EMER.,EMERGENCY ROOM,HOME,Medicaid,English,WIDOWED,WHITE,1.015278,1,25.740278,29079034.0,10000032-DS-22,2180-06-27 00:00:00,2180-07-01 10:15:00,\nName: ___ Unit No: _...
2,10000032,29079034,2180-07-23 12:35:00,2180-07-25 17:55:00,EW EMER.,EMERGENCY ROOM,HOME,Medicaid,English,WIDOWED,WHITE,2.222222,1,11.242361,25742920.0,10000032-DS-23,2180-07-25 00:00:00,2180-07-25 21:42:00,\nName: ___ Unit No: _...


In [ ]:
# Check if any hadm_id has multiple notes
duplicates = labeled_with_notes['hadm_id'].duplicated().sum()
print(f"\nDuplicate hadm_ids (admissions with multiple notes): {duplicates}")

if duplicates > 0:
    print("\nSome admissions have multiple discharge notes - we'll handle this...")
    # Count notes per admission
    notes_per_admission = labeled_with_notes.groupby('hadm_id').size()
    print(f"Admissions with >1 note: {(notes_per_admission > 1).sum()}")
    print(f"Max notes for single admission: {notes_per_admission.max()}")


Duplicate hadm_ids (admissions with multiple notes): 0


In [ ]:
# No duplicates - each admission has exactly one note
labeled_with_notes_clean = labeled_with_notes.copy()
print("No duplicates - each admission has exactly one note")
print(f"Final dataset: {len(labeled_with_notes_clean):,} admissions")

No duplicates - each admission has exactly one note
Final dataset: 311,459 admissions


In [ ]:
# Save the dataset with notes
output_file = OUTPUT_DIR / 'labeled_admissions_with_notes.csv'

labeled_with_notes_clean.to_csv(output_file, index=False)

print(f"\nSaved paired dataset to: {output_file}")
print(f"Final shape: {labeled_with_notes_clean.shape}")
print(f"Total admissions with labels and notes: {len(labeled_with_notes_clean):,}")


Saved paired dataset to: /content/drive/MyDrive/MIDS/w266/Final Project/output/labeled_admissions_with_notes.csv
Final shape: (311459, 19)
Total admissions with labels and notes: 311,459


In [ ]:
print("\n" + "="*60)
print("NOTEBOOK 2 COMPLETE: NOTES LINKED TO ADMISSIONS")
print("="*60)
print(f"\nFinal dataset summary:")
print(f"  Total admissions: {len(labeled_with_notes_clean):,}")
print(f"  Readmitted (Y=1): {(labeled_with_notes_clean['readmitted_30day']==1).sum():,}")
print(f"  Not readmitted (Y=0): {(labeled_with_notes_clean['readmitted_30day']==0).sum():,}")
print(f"  Readmission rate: {labeled_with_notes_clean['readmitted_30day'].mean()*100:.2f}%")
print(f"\nAverage note length: {labeled_with_notes_clean['text'].str.len().mean():.0f} characters")
print(f"\nOutput saved to: {output_file}")
print(f"\nReady for Notebook 3: Link timestamped events")


NOTEBOOK 2 COMPLETE: NOTES LINKED TO ADMISSIONS

Final dataset summary:
  Total admissions: 311,459
  Readmitted (Y=1): 64,309
  Not readmitted (Y=0): 247,150
  Readmission rate: 20.65%

Average note length: 10407 characters

Output saved to: /content/drive/MyDrive/MIDS/w266/Final Project/output/labeled_admissions_with_notes.csv

Ready for Notebook 3: Link timestamped events
